In [8]:
import pandas as pd
import re

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
location_df_normalized = pd.read_csv('../data/location_df_normalized.csv')

In [3]:
# Store original column names
original_columns = location_df_normalized.columns.tolist()

# Function to normalize column names
def normalize_column_name(column):
    
    # Convert to lowercase
    column = column.lower()
    
    # Remove leading and trailing spaces
    column = column.strip()
    
    # Replace punctuation with spaces
    column = re.sub(r'[-_.,]', ' ', column)
    
    # Replace multiple spaces with a single space
    column = re.sub(r'\s+', ' ', column)
    
    # Standardize common abbreviations
    replacements = {
        r'\brd\b': 'road',
        r'\bst\b': 'street',
        r'\bexpy\b': 'expressway',
        r'\bhwy\b': 'highway',
        r'\bintl\b': 'international'
    }
    
    for pattern, replacement in replacements.items():
        column = re.sub(pattern, replacement, column)
    
    return column.strip()


# Create normalized column names
normalized_columns = [
    normalize_column_name(col)
    for col in original_columns
]


# Create a mapping DataFrame
column_mapping_df = pd.DataFrame({
    'original_column': original_columns,
    'normalized_column': normalized_columns
})


# Display total number of columns
print("Total original columns:", len(original_columns))


# Display first 50 mappings
print("\nFirst 50 column mappings:")
print(column_mapping_df.head(50))


# Find duplicate normalized column names
duplicate_groups = (
    column_mapping_df
    .groupby('normalized_column')['original_column']
    .apply(list)
)


# Keep only groups with more than one original column
duplicate_groups = duplicate_groups[
    duplicate_groups.apply(len) > 1
]


# Print duplicate groups
print("\nDuplicate groups after normalization:\n")

for normalized_name, original_names in duplicate_groups.items():
    
    print(f"Normalized Name: {normalized_name}")
    print("Original columns:")
    
    for name in original_names:
        print(f"- {name}")
    
    print("-" * 50)

Total original columns: 1071

First 50 column mappings:
                           original_column  \
0                             PropertyName   
1                            Bajghera Road   
2                         Palam Vihar Halt   
3                         DPSG Palam Vihar   
4                            Park Hospital   
5                  Gurgaon Railway Station   
6                  The NorthCap University   
7                              Dwarka Expy   
8          Hyatt Place Gurgaon Udyog Vihar   
9          Dwarka Sector 21, Metro Station   
10                        Pacific D21 Mall   
11     Indira Gandhi International Airport   
12                        Hamoni Golf Camp   
13                    Fun N Food Waterpark   
14                          Accenture DDC5   
15               DPSG Palam Vihar Gurugram   
16              Park Hospital, Palam Vihar   
17        Palam Vihar Halt Railway Station   
18          Dwarka Sector 21 Metro Station   
19                      

In [6]:
from rapidfuzz import fuzz, process

# Get the unique normalized column names
unique_normalized_columns = column_mapping_df[
    'normalized_column'
].unique().tolist()

# Minimum similarity score
threshold = 85

# Store similar pairs
similar_pairs = []

# Compare every column with every other column
for i, col1 in enumerate(unique_normalized_columns):
    
    for col2 in unique_normalized_columns[i + 1:]:
        
        similarity = fuzz.token_sort_ratio(col1, col2)
        
        # Keep only highly similar pairs
        if similarity >= threshold:
            
            similar_pairs.append({
                'column_1': col1,
                'column_2': col2,
                'similarity': round(similarity, 2)
            })


# Convert results to DataFrame
similar_columns_df = pd.DataFrame(similar_pairs)

# Sort by similarity score
similar_columns_df = similar_columns_df.sort_values(
    by='similarity',
    ascending=False
).reset_index(drop=True)


# Display results
print("Number of similar column pairs:", len(similar_columns_df))

print("\nTop similar column pairs:\n")

print(similar_columns_df.head(100))

Number of similar column pairs: 206

Top similar column pairs:

                                     column_1  \
0                    vivanta dwarka new delhi   
1                    gurgaon delhi expressway   
2            sector 55 56 rapid metro station   
3                    rapid metro sector 55 56   
4                  tau devilal sports complex   
..                                        ...   
95                 golf course extension road   
96                 golf course extension road   
97                    gurgaon railway station   
98  lotus valley international school gurgaon   
99               holiday inn express gurugram   

                               column_2  similarity  
0              vivanta new delhi dwarka      100.00  
1              delhi gurgaon expressway      100.00  
2      rapid metro station sector 55 56      100.00  
3              sector 55 56 rapid metro      100.00  
4           tau devi lal sports complex       98.11  
..                     

In [9]:
print(similar_columns_df)

                                      column_1  \
0                     vivanta dwarka new delhi   
1                     gurgaon delhi expressway   
2             sector 55 56 rapid metro station   
3                     rapid metro sector 55 56   
4                   tau devilal sports complex   
5                     badshahpur sohna highway   
6                       delhi ajmer expressway   
7                       proposed metro station   
8                         the millenium school   
9                         golf course ext road   
10     badshahpur sohna road highway sector 48   
11                        golf course ext road   
12                         national highway 48   
13                           dwarka expressway   
14                              paras hospital   
15                             ektaa hospitals   
16                              rions hospital   
17                              aarvy hospital   
18                 street xavier's high school   


In [10]:
display(similar_columns_df)

,column_1,column_2,similarity
0,vivanta dwarka new delhi,vivanta new delhi dwarka,100.00
1,gurgaon delhi expressway,delhi gurgaon expressway,100.00
2,sector 55 56 rapid metro station,rapid metro station sector 55 56,100.00
3,rapid metro sector 55 56,sector 55 56 rapid metro,100.00
4,tau devilal sports complex,tau devi lal sports complex,98.11
5,badshahpur sohna highway,badshapur sohna highway,97.87
6,delhi ajmer expressway,delh ajmer expressway,97.67
7,proposed metro station,propose metro station,97.67
8,the millenium school,the millennium school,97.56
9,golf course ext road,golf course extn road,97.56


In [11]:
import re
import pandas as pd


# Function to extract numbers from a location name
def extract_numbers(text):
    return re.findall(r'\d+', text)


# Function to classify similar pairs
def classify_pair(row):
    
    col1 = row['column_1']
    col2 = row['column_2']
    similarity = row['similarity']
    
    # Extract numbers
    numbers_1 = extract_numbers(col1)
    numbers_2 = extract_numbers(col2)
    
    
    # RULE 1:
    # If both names contain numbers and the numbers are different,
    # they should NOT be automatically merged
    if numbers_1 and numbers_2:
        if set(numbers_1) != set(numbers_2):
            return "REJECT - Different numbers"
    
    
    # RULE 2:
    # Very high similarity
    if similarity >= 95:
        return "LIKELY SAME"
    
    
    # RULE 3:
    # High similarity but requires checking
    elif similarity >= 88:
        return "MANUAL REVIEW"
    
    
    # RULE 4:
    # Lower similarity
    else:
        return "LOW CONFIDENCE"


# Create a copy so original dataframe remains unchanged
review_df = similar_columns_df.copy()


# Apply classification
review_df['classification'] = review_df.apply(
    classify_pair,
    axis=1
)


# Show number of pairs in each category
print("Classification summary:\n")

print(
    review_df['classification']
    .value_counts()
)


# Display likely same locations
likely_same_df = review_df[
    review_df['classification'] == 'LIKELY SAME'
]

print("\n\nLIKELY SAME LOCATIONS:\n")

display(likely_same_df)


# Display manual review pairs
manual_review_df = review_df[
    review_df['classification'] == 'MANUAL REVIEW'
]

print("\n\nMANUAL REVIEW:\n")

display(manual_review_df)

Classification summary:

classification
LOW CONFIDENCE                66
MANUAL REVIEW                 63
REJECT - Different numbers    52
LIKELY SAME                   25
Name: count, dtype: int64


LIKELY SAME LOCATIONS:



,column_1,column_2,similarity,classification
0,vivanta dwarka new delhi,vivanta new delhi dwarka,100.00,LIKELY SAME
1,gurgaon delhi expressway,delhi gurgaon expressway,100.00,LIKELY SAME
2,sector 55 56 rapid metro station,rapid metro station sector 55 56,100.00,LIKELY SAME
3,rapid metro sector 55 56,sector 55 56 rapid metro,100.00,LIKELY SAME
4,tau devilal sports complex,tau devi lal sports complex,98.11,LIKELY SAME
5,badshahpur sohna highway,badshapur sohna highway,97.87,LIKELY SAME
6,delhi ajmer expressway,delh ajmer expressway,97.67,LIKELY SAME
7,proposed metro station,propose metro station,97.67,LIKELY SAME
8,the millenium school,the millennium school,97.56,LIKELY SAME
9,golf course ext road,golf course extn road,97.56,LIKELY SAME




MANUAL REVIEW:



,column_1,column_2,similarity,classification
31,aapnoghar,aapno ghar,94.74,MANUAL REVIEW
32,the signature super speciality hospital,signature super speciality hospital,94.59,MANUAL REVIEW
34,country inn & suites by radisson,country inn and suites by radisson,93.94,MANUAL REVIEW
35,vardaan hospital and trauma centre,vardaan hospital & trauma centre,93.94,MANUAL REVIEW
42,artimis hospital,artemis hospital,93.75,MANUAL REVIEW
43,huda city centre,huda city center,93.75,MANUAL REVIEW
46,cd international school,dps international school,93.62,MANUAL REVIEW
47,southern peripheral road,southern periphery road,93.62,MANUAL REVIEW
49,nh 248a,nh 248 a,93.33,MANUAL REVIEW
51,gd goenka world school,g d goenka world school,93.33,MANUAL REVIEW


In [12]:
from collections import defaultdict

# --------------------------------------------------
# STEP 4: CREATE CONFIRMED DUPLICATE PAIRS
# --------------------------------------------------

# Take all LIKELY SAME pairs
confirmed_pairs = []

likely_same = review_df[
    review_df['classification'] == 'LIKELY SAME'
]

for _, row in likely_same.iterrows():
    confirmed_pairs.append(
        (row['column_1'], row['column_2'])
    )


# --------------------------------------------------
# ADD MANUALLY APPROVED PAIRS
# --------------------------------------------------

manual_confirmed_pairs = [

    ('aapnoghar', 'aapno ghar'),

    ('the signature super speciality hospital',
     'signature super speciality hospital'),

    ('country inn & suites by radisson',
     'country inn and suites by radisson'),

    ('vardaan hospital and trauma centre',
     'vardaan hospital & trauma centre'),

    ('artimis hospital', 'artemis hospital'),

    ('huda city centre', 'huda city center'),

    ('nh 248a', 'nh 248 a'),

    ('gd goenka world school',
     'g d goenka world school'),

    ('kr mangalam university',
     'k r mangalam university'),

    ('sanjivani hospital', 'sanjeevani hospital'),

    ('indira gandhi int airport',
     'indira gandhi airport'),

    ('hyatt regency gurugram',
     'hyatt regency gurgaon'),

    ('gurgaon railway station',
     'gurugram railway station'),

    ('golf course extension road',
     'golf course extn road'),

    ('golf course extension road',
     'golf course extension'),

    ('lotus valley international school gurgaon',
     'lotus valley international school'),

    ('holiday inn express gurugram',
     'holiday inn express gurugram sec 50'),

    ('euro international school sector 37d',
     'euro international school sector 37d gurugram'),

    ('badshahpur sohna highway',
     'badshahpur sohna road highway'),

    ('miracles apollo cradle hospital',
     'miracles apollo cradle spectra hospital'),

    ('miracles apollo cradle hospital',
     'miracles apollo cradle/spectra hospital'),

    ('nh 8 delhi jaipur highway',
     'delhi jaipur highway'),

    ('g d goenka university',
     'gd goenka university')
]


# Add manual pairs
confirmed_pairs.extend(manual_confirmed_pairs)


# --------------------------------------------------
# CREATE CONNECTED DUPLICATE GROUPS
# --------------------------------------------------

graph = defaultdict(set)

for col1, col2 in confirmed_pairs:
    graph[col1].add(col2)
    graph[col2].add(col1)


# Find connected components
visited = set()
duplicate_groups = []

for node in graph:

    if node not in visited:

        stack = [node]
        group = []

        while stack:

            current = stack.pop()

            if current not in visited:

                visited.add(current)
                group.append(current)

                for neighbor in graph[current]:

                    if neighbor not in visited:
                        stack.append(neighbor)

        duplicate_groups.append(sorted(group))


# --------------------------------------------------
# PRINT DUPLICATE GROUPS
# --------------------------------------------------

print("Total duplicate groups:", len(duplicate_groups))

for i, group in enumerate(duplicate_groups, start=1):

    print(f"\nGROUP {i}")

    for column in group:
        print(f"- {column}")

    print("-" * 50)

Total duplicate groups: 41

GROUP 1
- vivanta dwarka new delhi
- vivanta new delhi dwarka
--------------------------------------------------

GROUP 2
- delhi gurgaon expressway
- gurgaon delhi expressway
--------------------------------------------------

GROUP 3
- rapid metro station sector 55 56
- sector 55 56 rapid metro station
--------------------------------------------------

GROUP 4
- rapid metro sector 55 56
- sector 55 56 rapid metro
--------------------------------------------------

GROUP 5
- tau devi lal sports complex
- tau devilal sports complex
--------------------------------------------------

GROUP 6
- badshahpur sohna highway
- badshahpur sohna road highway
- badshapur sohna highway
--------------------------------------------------

GROUP 7
- delh ajmer expressway
- delhi ajmer expressway
--------------------------------------------------

GROUP 8
- propose metro station
- proposed metro station
--------------------------------------------------

GROUP 9
- the mill

In [13]:
# --------------------------------------------------
# STEP 5: MAP DUPLICATE GROUPS TO ORIGINAL COLUMNS
# AND MERGE THEM
# --------------------------------------------------

# Create a copy so the original dataframe is safe
location_df_cleaned = location_df_normalized.copy()


# --------------------------------------------------
# Function to get all original columns
# belonging to a normalized column name
# --------------------------------------------------

def get_original_columns(normalized_name):
    
    return column_mapping_df[
        column_mapping_df['normalized_column'] == normalized_name
    ]['original_column'].tolist()


# --------------------------------------------------
# Store merge information
# --------------------------------------------------

merge_summary = []


# --------------------------------------------------
# Process each duplicate group
# --------------------------------------------------

for group_number, group in enumerate(duplicate_groups, start=1):
    
    # Get all actual/original column names
    columns_to_merge = []
    
    for normalized_name in group:
        
        original_cols = get_original_columns(normalized_name)
        
        columns_to_merge.extend(original_cols)
    
    
    # Remove duplicates
    columns_to_merge = list(set(columns_to_merge))
    
    
    # Keep only columns that actually exist
    columns_to_merge = [
        col for col in columns_to_merge
        if col in location_df_cleaned.columns
    ]
    
    
    # Skip if fewer than 2 columns exist
    if len(columns_to_merge) < 2:
        continue
    
    
    # --------------------------------------------------
    # Choose a canonical column name
    # Here we use the first normalized name
    # --------------------------------------------------
    
    canonical_name = group[0]
    
    
    # --------------------------------------------------
    # Merge distances
    # Take the minimum available distance
    # --------------------------------------------------
    
    location_df_cleaned[canonical_name] = (
        location_df_cleaned[columns_to_merge]
        .min(axis=1, skipna=True)
    )
    
    
    # --------------------------------------------------
    # Drop old duplicate columns
    # BUT don't drop canonical_name if it already exists
    # --------------------------------------------------
    
    columns_to_drop = [
        col for col in columns_to_merge
        if col != canonical_name
    ]
    
    location_df_cleaned.drop(
        columns=columns_to_drop,
        inplace=True,
        errors='ignore'
    )
    
    
    # Store summary
    merge_summary.append({
        'group': group_number,
        'canonical_name': canonical_name,
        'merged_columns': columns_to_merge,
        'number_of_columns_merged': len(columns_to_merge)
    })


# --------------------------------------------------
# CREATE MERGE SUMMARY DATAFRAME
# --------------------------------------------------

merge_summary_df = pd.DataFrame(merge_summary)


# --------------------------------------------------
# RESULTS
# --------------------------------------------------

print("Original number of columns:",
      location_df_normalized.shape[1])

print("Cleaned number of columns:",
      location_df_cleaned.shape[1])

print("Columns reduced by:",
      location_df_normalized.shape[1]
      - location_df_cleaned.shape[1])


print("\nMerge Summary:\n")

display(
    merge_summary_df[
        ['group',
         'canonical_name',
         'number_of_columns_merged',
         'merged_columns']
    ]
)

Original number of columns: 1071
Cleaned number of columns: 1010
Columns reduced by: 61

Merge Summary:



,group,canonical_name,number_of_columns_merged,merged_columns
0,1,vivanta dwarka new delhi,2,"[Vivanta Dwarka New Delhi, Vivanta New Delhi, ..."
1,2,delhi gurgaon expressway,2,"[Delhi Gurgaon Expressway, Gurgaon - Delhi Expy]"
2,3,rapid metro station sector 55 56,3,"[Rapid Metro Station Sector 55 56, Sector 55-5..."
3,4,rapid metro sector 55 56,2,"[Sector 55-56 Rapid Metro, Rapid Metro Sector ..."
4,5,tau devi lal sports complex,2,"[Tau Devi Lal Sports Complex, Tau DeviLal Spor..."
5,6,badshahpur sohna highway,3,"[Badshahpur Sohna Hwy, Badshapur Sohna Highway..."
6,7,delh ajmer expressway,2,"[Delh-Ajmer Expy, Delhi Ajmer Expressway]"
7,8,propose metro station,2,"[Proposed Metro Station, Propose Metro Station]"
8,9,the millenium school,2,"[The Millennium School, The Millenium School]"
9,10,golf corse ext road,8,"[Golf course extension road, Golf Course Exten..."


In [17]:
location_df_cleaned.columns.tolist()

['PropertyName',
 'Bajghera Road',
 'Palam Vihar Halt',
 'DPSG Palam Vihar',
 'Park Hospital',
 'The NorthCap University',
 'Hyatt Place Gurgaon Udyog Vihar',
 'Dwarka Sector 21, Metro Station',
 'Pacific D21 Mall',
 'Indira Gandhi International Airport',
 'Hamoni Golf Camp',
 'Fun N Food Waterpark',
 'Accenture DDC5',
 'DPSG Palam Vihar Gurugram',
 'Park Hospital, Palam Vihar',
 'Palam Vihar Halt Railway Station',
 'Dwarka Sector 21 Metro Station',
 'Fun N Food Water Park',
 'Hyatt Place',
 'Altrade Business Centre',
 'AIPL Business Club Sector 62',
 'Heritage Xperiential Learning School',
 'CK Birla Hospital',
 'Paras Trinity Mall Sector 63',
 'Rapid Metro Station Sector 56',
 'De Adventure Park',
 'DoubleTree by Hilton Hotel Gurgaon',
 'KIIT College of Engineering Sohna Road',
 'Mehrauli-Gurgaon Road',
 'Nirvana Rd',
 'TERI Golf Course',
 'The Shikshiyan School',
 'WTC Plaza',
 'Luxus Haritma Resort',
 'BSF Golf Course',
 'Gurgaon',
 'Dwarka Sector 21',
 'Nehru Stadium',
 'Fun N Foo

In [18]:
from rapidfuzz import fuzz
import pandas as pd
import re

# --------------------------------------------------
# FINAL CHECK FOR POSSIBLE DUPLICATE COLUMNS
# --------------------------------------------------

# Get all remaining column names
columns = location_df_cleaned.columns.tolist()


# --------------------------------------------------
# Function to normalize names again for comparison
# --------------------------------------------------

def normalize_for_check(name):
    
    name = name.lower().strip()
    
    # Replace punctuation with spaces
    name = re.sub(r'[-_.,/&]', ' ', name)
    
    # Remove multiple spaces
    name = re.sub(r'\s+', ' ', name)
    
    return name.strip()


# Normalize remaining column names
normalized_remaining = {
    col: normalize_for_check(col)
    for col in columns
}


# --------------------------------------------------
# Find similar pairs
# --------------------------------------------------

threshold = 75

remaining_similar_pairs = []

for i, col1 in enumerate(columns):
    
    for col2 in columns[i + 1:]:
        
        name1 = normalized_remaining[col1]
        name2 = normalized_remaining[col2]
        
        similarity = fuzz.token_sort_ratio(name1, name2)
        
        if similarity >= threshold:
            
            remaining_similar_pairs.append({
                'column_1': col1,
                'column_2': col2,
                'normalized_1': name1,
                'normalized_2': name2,
                'similarity': round(similarity, 2)
            })


# --------------------------------------------------
# Create results DataFrame
# --------------------------------------------------

final_check_df = pd.DataFrame(remaining_similar_pairs)

# Sort highest similarity first
final_check_df = final_check_df.sort_values(
    by='similarity',
    ascending=False
).reset_index(drop=True)


# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------

print("Total remaining columns:", len(columns))

print("\nPossible similar pairs found:", len(final_check_df))

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display(final_check_df)

Total remaining columns: 1010

Possible similar pairs found: 915


,column_1,column_2,normalized_1,normalized_2,similarity
0,Matrikiran High School,MatriKiran High School,matrikiran high school,matrikiran high school,100.00
1,Dwarka Sector 21 Metro station,Dwarka sector 21 metro station,dwarka sector 21 metro station,dwarka sector 21 metro station,100.00
2,Sector 42-43 Metro Station,Sector 42-43 Metro station,sector 42 43 metro station,sector 42 43 metro station,100.00
3,"Delhi Public School, Sector 84",Delhi Public School Sector 84,delhi public school sector 84,delhi public school sector 84,100.00
4,NH 48,NH-48,nh 48,nh 48,100.00
5,Global city centre,Global City Centre,global city centre,global city centre,100.00
6,Sector 55-56 metro,Sector 55-56 Metro,sector 55 56 metro,sector 55 56 metro,100.00
7,HUB 66,Hub 66,hub 66,hub 66,100.00
8,The Oberoi Gurgaon,"The Oberoi, Gurgaon",the oberoi gurgaon,the oberoi gurgaon,100.00
9,"International Tech Park Gurgaon,",International Tech Park Gurgaon,international tech park gurgaon,international tech park gurgaon,100.00


In [25]:
final_check_df.to_csv(
    "final_duplicate_check.csv",
    index=False
)

In [26]:
# --------------------------------------------------
# STEP 6: FIND EXACT DUPLICATES AFTER NORMALIZATION
# --------------------------------------------------

from collections import defaultdict

# Get current columns
columns = location_df_cleaned.columns.tolist()


# Create groups based on normalized names
exact_duplicate_groups = defaultdict(list)

for col in columns:
    
    normalized_name = normalize_for_check(col)
    
    exact_duplicate_groups[normalized_name].append(col)


# Keep only groups having more than 1 column
exact_duplicate_groups = {
    normalized: cols
    for normalized, cols in exact_duplicate_groups.items()
    if len(cols) > 1
}


# Print results
print("Number of exact duplicate groups:",
      len(exact_duplicate_groups))


for normalized_name, cols in exact_duplicate_groups.items():
    
    print(f"\nNORMALIZED NAME: {normalized_name}")
    
    for col in cols:
        print("-", col)
    
    print("-" * 50)

Number of exact duplicate groups: 42

NORMALIZED NAME: dwarka sector 21 metro station
- Dwarka Sector 21, Metro Station
- Dwarka Sector 21 Metro Station
- Dwarka Sector 21 Metro station
- Dwarka sector 21 metro station
--------------------------------------------------

NORMALIZED NAME: fun n food waterpark
- Fun N Food Waterpark
- Fun N Food WaterPark
--------------------------------------------------

NORMALIZED NAME: teri golf course
- TERI Golf Course
- Teri Golf Course
--------------------------------------------------

NORMALIZED NAME: nh 48
- NH 48
- NH-48
--------------------------------------------------

NORMALIZED NAME: nh 8
- NH -8
- NH-8
- NH 8
--------------------------------------------------

NORMALIZED NAME: imt manesar
- IMT Manesar
- Imt Manesar
--------------------------------------------------

NORMALIZED NAME: euro international school sector 109
- Euro International School, Sector- 109
- Euro International School, Sector- 109.
------------------------------------

In [27]:
from collections import defaultdict

# Create a copy of the current cleaned dataframe
location_df_final = location_df_cleaned.copy()

# Store merge details
final_merge_summary = []


# Loop through every exact duplicate group
for normalized_name, columns_to_merge in exact_duplicate_groups.items():
    
    # Keep only columns that currently exist
    existing_columns = [
        col for col in columns_to_merge
        if col in location_df_final.columns
    ]
    
    
    # Skip if fewer than 2 columns exist
    if len(existing_columns) < 2:
        continue
    
    
    # Use normalized name as the final canonical column name
    canonical_name = normalized_name
    
    
    # Merge using minimum available distance
    location_df_final[canonical_name] = (
        location_df_final[existing_columns]
        .min(axis=1, skipna=True)
    )
    
    
    # Drop all original duplicate columns
    location_df_final.drop(
        columns=existing_columns,
        inplace=True,
        errors='ignore'
    )
    
    
    # Save merge information
    final_merge_summary.append({
        'canonical_name': canonical_name,
        'number_of_columns_merged': len(existing_columns),
        'merged_columns': existing_columns
    })


# Convert summary to DataFrame
final_merge_summary_df = pd.DataFrame(final_merge_summary)


# Print results
print("Columns before final exact-duplicate merge:",
      location_df_cleaned.shape[1])

print("Columns after final exact-duplicate merge:",
      location_df_final.shape[1])

print("Total columns reduced:",
      location_df_cleaned.shape[1] - location_df_final.shape[1])


# Display merge summary
display(final_merge_summary_df)

Columns before final exact-duplicate merge: 1010
Columns after final exact-duplicate merge: 959
Total columns reduced: 51


,canonical_name,number_of_columns_merged,merged_columns
0,dwarka sector 21 metro station,4,"[Dwarka Sector 21, Metro Station, Dwarka Secto..."
1,fun n food waterpark,2,"[Fun N Food Waterpark, Fun N Food WaterPark]"
2,teri golf course,2,"[TERI Golf Course, Teri Golf Course]"
3,nh 48,2,"[NH 48, NH-48]"
4,nh 8,3,"[NH -8, NH-8, NH 8]"
5,imt manesar,2,"[IMT Manesar, Imt Manesar]"
6,euro international school sector 109,2,"[Euro International School, Sector- 109, Euro ..."
7,pataudi road,2,"[Pataudi Road, Pataudi road]"
8,iris broadway mall,2,"[Iris Broadway Mall, IRIS Broadway Mall]"
9,omaxe gurgaon mall,2,"[Omaxe Gurgaon Mall, OMAXE Gurgaon Mall]"


In [31]:
location_df_final.shape

(246, 959)